# ML Coding — Day 8: Quantization II (PTQ & calibration)

**~1 hour, vectorized NumPy.** Post-training quantization: choosing scales (percentile, MSE), the int8
GEMM with zero-point correction, and dynamic vs static activation quant. Forward-only (PTQ has no
training); the stretch is the fused zero-point algebra. No element loops in solutions. Run the
**helpers cell first**.

**Q1** percentile calibration `[warmup]` · **Q2** MSE-optimal scale `[medium]` · **Q3** int8 GEMM
`[medium]` · **Q4** dynamic vs static `[hard]` · **Q5** fused requant (zero-point trick) `[hard · trick]`.

---

In [ ]:
# --- helpers (run me first) ---
import numpy as np

## Q1 — Percentile calibration  ·  `[warmup]`

Min/max calibration is wrecked by a single outlier. `percentile_qparams(x, qmax, pct) -> scale`
(symmetric, zero-point 0): clip the range at the `pct`-th percentile of `|x|`, then `scale = clip / qmax`.

**Lock down:** clipping the tail trades a few saturated outliers for far better resolution on the bulk
of the distribution — the simplest robust calibrator.

In [ ]:
import numpy as np


def percentile_qparams(x, qmax, pct):
    raise NotImplementedError

In [ ]:
# --- Q1 tests ---
def _q1():
    x = np.concatenate([np.ones(1000), np.array([1000.0])])      # one big outlier
    s_full = np.max(np.abs(x)) / 127
    s_pct = percentile_qparams(x, 127, 99.0)
    assert s_pct < s_full                                         # outlier ignored -> finer scale
    assert np.isclose(s_pct, 1.0 / 127, atol=0.01)
    print("Q1 OK")

_q1()

## Q2 — MSE-optimal scale  ·  `[medium]`

`mse_optimal_scale(x, qmax, candidates) -> scale`: pick the candidate scale minimizing the quantization
MSE `mean((x − dequant(quant(x)))²)`, where `quant(x) = clip(round(x/s), −qmax, qmax)`. Evaluate **all**
candidates at once (vectorized, no Python loop over them).

**Lock down:** build a `(C, N)` matrix of dequantized values (`candidates × x`), take the per-candidate
MSE, `argmin`. MSE-optimal usually clips slightly tighter than min/max.

In [ ]:
def mse_optimal_scale(x, qmax, candidates):
    raise NotImplementedError

In [ ]:
# --- Q2 tests ---
def _q2():
    rng = np.random.default_rng(0)
    x = rng.standard_normal(1000)
    maxabs = np.max(np.abs(x))
    cands = np.linspace(maxabs / 127, 3 * maxabs / 127, 50)
    best = mse_optimal_scale(x, 127, cands)

    def mse(s):
        return np.mean((x - np.clip(np.round(x / s), -127, 127) * s) ** 2)

    assert mse(best) <= mse(maxabs / 127) + 1e-12               # no worse than the min/max scale
    print("Q2 OK")

_q2()

## Q3 — Int8 GEMM with zero-point  ·  `[medium]`

`quantized_matmul(A_int, sa, za, W_int, sw) -> Y_float`: activations are asymmetric int (`scale sa`,
`zero_point za`), weights are **per-output-channel** symmetric int (`sw` a vector). Compute
`Y = sa · sw · ((A_int − za) @ W_int)` with **integer accumulation** (int32/64), then dequantize.

**Lock down:** the matmul runs in integers (that's the speed win); `sw` broadcasts over output columns;
this is the kernel under int8 inference.

In [ ]:
def quantized_matmul(A_int, sa, za, W_int, sw):
    raise NotImplementedError

In [ ]:
# --- Q3 tests ---
def _q3():
    rng = np.random.default_rng(1)
    A = rng.standard_normal((4, 8)); W = rng.standard_normal((8, 5))
    sa = (A.max() - A.min()) / 255
    za = int(round(-A.min() / sa))
    A_int = np.clip(np.round(A / sa) + za, 0, 255).astype(int)
    sw = np.max(np.abs(W), axis=0) / 127
    W_int = np.clip(np.round(W / sw), -127, 127).astype(int)
    Y = quantized_matmul(A_int, sa, za, W_int, sw)
    assert Y.shape == (4, 5)
    assert np.max(np.abs(Y - A @ W)) < 0.5                       # close to the float matmul
    print("Q3 OK")

_q3()

## Q4 — Dynamic vs static activation quant  ·  `[hard]`

`quantize_linear(x, W, act_scale=None, qmax=127) -> Y_float`: quantize activations (symmetric) — either
**dynamic** (compute the scale per call from `x`'s range, when `act_scale is None`) or **static** (use
the passed pre-calibrated `act_scale`) — quantize `W` per output channel, int-matmul, dequantize.

**The point:** a static scale calibrated on small inputs **clips** a larger input → big error; dynamic
adapts. Verify dynamic beats a stale static scale on a 10× input. No loops.

In [ ]:
def quantize_linear(x, W, act_scale=None, qmax=127):
    raise NotImplementedError

In [ ]:
# --- Q4 tests ---
def _q4():
    rng = np.random.default_rng(2)
    x = rng.standard_normal((4, 8)); W = rng.standard_normal((8, 5))
    assert np.max(np.abs(quantize_linear(x, W) - x @ W)) < 0.5    # dynamic is accurate
    stale = np.max(np.abs(x)) / 127                               # calibrated on the small input
    x_big = x * 10
    err_static = np.max(np.abs(quantize_linear(x_big, W, act_scale=stale) - x_big @ W))
    err_dyn = np.max(np.abs(quantize_linear(x_big, W) - x_big @ W))
    assert err_dyn < err_static                                   # dynamic adapts, static clips
    print("Q4 OK")

_q4()

## Q5 — Fused requant: the zero-point trick  ·  `[hard · tensor trick]`

Subtracting `za` from the whole activation matrix before the GEMM is wasteful. Use the identity
`(A_int − za) @ W_int = A_int @ W_int − za · colsum(W_int)`: run the integer GEMM on raw `A_int`, then
correct with a **precomputed per-column weight sum** — one cheap vector subtraction.

Implement `fused_requant(A_int, sa, za, W_int, sw) -> Y_float`; it must equal `quantized_matmul`.

**The trick:** the correction term `za · Σ_rows W_int` is precomputed once per weight matrix (offline),
so the runtime kernel never materializes `A_int − za`. No loops.

In [ ]:
def fused_requant(A_int, sa, za, W_int, sw):
    raise NotImplementedError

In [ ]:
# --- Q5 tests ---
def _q5():
    rng = np.random.default_rng(3)
    A_int = rng.integers(0, 256, (4, 8)); W_int = rng.integers(-127, 128, (8, 5))
    sa, za = 0.02, 128
    sw = rng.uniform(0.01, 0.05, 5)
    assert np.allclose(fused_requant(A_int, sa, za, W_int, sw),
                       quantized_matmul(A_int, sa, za, W_int, sw))
    print("Q5 OK")

_q5()

## Likely follow-ups
- Bias correction; cross-layer equalization; GPTQ/AWQ (Day 9) for weight-only quant.
- Symmetric vs asymmetric trade-offs; per-tensor vs per-channel vs per-group scales.
- Requantization to int8 between layers (fixed-point multiplier + shift) for integer-only inference.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import numpy as np


def ref_percentile_qparams(x, qmax, pct):
    x = np.asarray(x, float).ravel()
    hi = np.percentile(np.abs(x), pct)
    return hi / qmax if hi > 0 else 1.0


def ref_mse_optimal_scale(x, qmax, candidates):
    x = np.asarray(x, float).ravel()[None, :]
    s = np.asarray(candidates, float)[:, None]
    xh = np.clip(np.round(x / s), -qmax, qmax) * s
    mse = np.mean((x - xh) ** 2, axis=1)
    return np.asarray(candidates)[int(np.argmin(mse))]


def ref_quantized_matmul(A_int, sa, za, W_int, sw):
    A_int = np.asarray(A_int).astype(np.int64)
    W_int = np.asarray(W_int).astype(np.int64)
    acc = (A_int - za) @ W_int
    return sa * acc * np.asarray(sw)[None, :]


def ref_quantize_linear(x, W, act_scale=None, qmax=127):
    x = np.asarray(x, float); W = np.asarray(W, float)
    if act_scale is None:
        act_scale = np.max(np.abs(x)) / qmax
        act_scale = act_scale if act_scale > 0 else 1.0
    x_int = np.clip(np.round(x / act_scale), -qmax, qmax)
    sw = np.max(np.abs(W), axis=0) / qmax
    sw = np.where(sw > 0, sw, 1.0)
    w_int = np.clip(np.round(W / sw), -qmax, qmax)
    return act_scale * (x_int @ w_int) * sw[None, :]


def ref_fused_requant(A_int, sa, za, W_int, sw):
    A_int = np.asarray(A_int).astype(np.int64)
    W_int = np.asarray(W_int).astype(np.int64)
    acc = A_int @ W_int                                  # integer GEMM on raw activations
    acc = acc - za * W_int.sum(axis=0)[None, :]          # precomputed zero-point correction
    return sa * acc * np.asarray(sw)[None, :]


_x = np.concatenate([np.ones(100), [500.0]])
assert ref_percentile_qparams(_x, 127, 99.0) < np.max(np.abs(_x)) / 127
_xx = np.random.default_rng(4).standard_normal(500); _ma = np.max(np.abs(_xx))
_best = ref_mse_optimal_scale(_xx, 127, np.linspace(_ma / 127, 2 * _ma / 127, 20))
assert _best > 0
_Aint = np.random.default_rng(5).integers(0, 256, (3, 4)); _Wint = np.random.default_rng(6).integers(-127, 128, (4, 2))
assert np.allclose(ref_fused_requant(_Aint, 0.02, 128, _Wint, np.array([0.03, 0.04])),
                   ref_quantized_matmul(_Aint, 0.02, 128, _Wint, np.array([0.03, 0.04])))
print("reference solutions OK")